In [0]:
%pip install xgboost scikit-learn
dbutils.library.restartPython()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.5/252.5 MB 225.4 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import mlflow
import mlflow.xgboost
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
from pyspark.sql import SparkSession
from mlflow.models import infer_signature

# 1. Load Silver data from Unity Catalog / Delta Lake
silver_df = spark.table("workspace.default.stg_patient_appointments").toPandas()

# 2. Select predictive features & target
feature_cols = [
    'age', 'has_scholarship', 'has_hypertension', 
    'has_diabetes', 'has_alcoholism', 'disability_level', 
    'sms_reminder_sent', 'waiting_days'
]
X = silver_df[feature_cols]
y = silver_df['is_noshow']

# 3. Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Enable MLflow tracking
mlflow.set_experiment("/Users/" + spark.sql("SELECT current_user()").collect()[0][0] + "/patient_noshow_prediction")

with mlflow.start_run(run_name="xgboost_baseline_model"):
    # Hyperparameters
    params = {
        "n_estimators": 100,
        "max_depth": 5,
        "learning_rate": 0.1,
        "random_state": 42
    }
    
    # Train model
    model = XGBClassifier(**params)
    model.fit(X_train, y_train)
    
    # Predictions
    preds = model.predict(X_test)
    probs = model.predict_proba(X_test)[:, 1]
    
    # Evaluate metrics
    acc = accuracy_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    f1 = f1_score(y_test, preds)
    
# Infer input and output schema for Unity Catalog compliance
    signature = infer_signature(X_train, preds)
    
    # Log parameters, metrics, and signed model to MLflow
    mlflow.log_params(params)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("auc_roc", auc)
    mlflow.log_metric("f1_score", f1)
    
    mlflow.xgboost.log_model(
        xgb_model=model, 
        name="noshow_xgboost_model",
        signature=signature,
        registered_model_name="workspace.default.patient_noshow_model"
    )
    
    print(f"Model Training Complete! Accuracy: {acc:.4f} | AUC-ROC: {auc:.4f} | F1: {f1:.4f}")

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
🔗 View Logged Model at: https://dbc-5174e37a-f25f.cloud.databricks.com/ml/experiments/4062817422184168/models/m-8068ed6197e249b094223ee58102046a?o=7474650995543354
Registered model 'workspace.default.patient_noshow_model' already exists. Cr

Uploading artifacts:   0%|          | 0/9 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.patient_noshow_model': https://dbc-5174e37a-f25f.cloud.databricks.com/explore/data/models/workspace/default/patient_noshow_model/version/1?o=7474650995543354


Model Training Complete! Accuracy: 0.7986 | AUC-ROC: 0.7247 | F1: 0.0185
